# Optimal transport-based domain adaptation in wheat heads

This notebook focuses on preprocessing the images from the GWHD 2021 dataset, using domain adaptation based on Sinkhorn divergences with the aid of the GeomLoss package. This process creates a copy of the dataset where all the images have a similar color palette, which improves the detection metrics of a YOLO model.

## Preparing the images for the YOLO model

By executing the 'setup.py' script, we create the correct directory structure, as well as the neccessary label files in the YOLO format, which are copied to the folders where the modified dataset copy will be stored. This allows us to use the Ultralytics module to train and validate the YOLO model. Finally, 'setup.py' also creates YAML files to indicate Ultralytics where the images and its labels are located.

In [ ]:
!python 'setup.py'

## Perform the OT domain adaptation on the images and save them to the new folders

In [1]:
# Import the necessary libraries and modules for the project 
import os
import pandas as pd
import numpy as np
import random
import imageio
import torch
from geomloss import SamplesLoss
from PIL import Image
from utils import RGB_cloud, color_transfer, copy_files


In [2]:
# Ensures GPU support is available and use cuda tensors in this case 
use_cuda = torch.cuda.is_available()
dtype = torch.cuda.FloatTensor if use_cuda else torch.FloatTensor
sampling = 8 if not use_cuda else 1

# Function to apply domain adaptation iteratively to images from the source domains (all the images that don't belong to target domain), 
# choosing a random image from the target domain in every iteration and using Sinkhorn divergences as the loss function for the color transfer function
def OT_and_save(source_domain_folder, img_destination_folder, img_list):
    for img in img_list:
        source_image_path = os.path.join(source_domain_folder, img) 
        target_image_path = os.path.join(train_folder, random.choice(target_list))

        X_i = RGB_cloud(source_image_path, sampling, dtype)
        Y_j = RGB_cloud(target_image_path, sampling, dtype)
        
        # Blur and reach relate to ε (entropy regularization) and ρ (unbalanced transport) respectively, these parameters regulate the domain adaptation
        # changing ε and ρ creates images that may or may not improve YOLO performance metrics   
        new_cloud = color_transfer(X_i, Y_j, SamplesLoss("sinkhorn", blur=0.6, reach=None))
        
        # The new RGB cloud must be converted to a uint8 image so it can be saved
        W = int(np.sqrt(len(new_cloud)))
        img_matrix = new_cloud.view(W, W, 3).detach().cpu().numpy()
        new_image = np.clip(img_matrix, 0, 1)
        final_image = (new_image * 255).astype(np.uint8)

        # Images are saved to be used later by the YOLO model
        save_path = os.path.join(img_destination_folder, img)
        imageio.imwrite(save_path, final_image)
        print(f'New OT image {img} successfully created at {img_destination_folder}')

In [6]:
# Path of the csv files that contain data for the train, valid and test splits
csv_train_path = "../gwhd_2021/competition_train.csv"
csv_valid_path = "../gwhd_2021/competition_val.csv"
csv_test_path = "../gwhd_2021/competition_test.csv"

# Read the csv files
train = pd.read_csv(csv_train_path) 
valid = pd.read_csv(csv_valid_path)
test = pd.read_csv(csv_test_path) 

# Print the names of the domains in the training set and the number of images in each of them, so we can choose a target domain in the next step
train.value_counts("domain", normalize=False)

domain
ETHZ_1            747
Arvalis_3         588
Arvalis_5         448
Rres_1            432
Arvalis_2         401
Arvalis_4         204
Inrae_1           176
Arvalis_6         160
NMBU_2             98
NMBU_1             82
Arvalis_1          66
Arvalis_11         60
Arvalis_10         60
Arvalis_9          32
ULiège-GxABT_1     30
Arvalis_12         29
Arvalis_7          24
Arvalis_8          20
Name: count, dtype: int64

In [ ]:
# Create a list of files that belong to the chosen target domain so we can transport the other images to this domain
target_list = train[train["domain"] == "ETHZ_1"]["image_name"].tolist()

# Create lists of files from the source domains, i.e. every domain except target domain
source_train_list = train[train["domain"] != "ETHZ_1"]["image_name"].tolist()
source_valid_list = valid["image_name"].tolist()
source_test_list = test["image_name"].tolist()

# Defines the folders where the original images are stored
train_folder = "../gwhd_2021/Original/train/images/"
valid_folder = "../gwhd_2021/Original/valid/images/"
test_folder = "../gwhd_2021/Original/test/images/"

# Path to destination folders for the images after OT is applied to them
train_img_destination = "../gwhd_2021/OT/train/images/"
valid_img_destination = "../gwhd_2021/OT/valid/images/"
test_img_destination = "../gwhd_2021/OT/test/images/"

# Create destination folders in case they don't exist
os.makedirs(train_img_destination, exist_ok=True)
os.makedirs(valid_img_destination, exist_ok=True)
os.makedirs(test_img_destination, exist_ok=True)


In [ ]:
# Make a copy of the target domain images in the new folder, as we're not applying OT to them
copy_files(train_folder, target_list, train_img_destination)

In [ ]:
# Set a seed so we can reproduce the OT results
random.seed(0)

# Performs optimal transport on every image and stores them in the corresponding folders
OT_and_save(train_folder, train_img_destination, source_train_list)
OT_and_save(valid_folder, valid_img_destination, source_valid_list)
OT_and_save(test_folder, test_img_destination, source_test_list)